In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
from torch.cuda.amp import GradScaler, autocast

# Load data
data = pd.read_excel('/content/Qualitative_coding_emotional_needs_TLR_LH.xlsx')


In [ ]:
# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Preprocess data
class PostsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Use DistilBert
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
def tokenize_function(posts):
    return tokenizer(posts, padding=True, truncation=True, max_length=128)

X_train, X_test, y_train, y_test = train_test_split(data['question_content/Original Post'], data['Emotional support need'], test_size=0.2)
train_encodings = tokenize_function(X_train.tolist())
test_encodings = tokenize_function(X_test.tolist())

train_dataset = PostsDataset(train_encodings, y_train.tolist())
test_dataset = PostsDataset(test_encodings, y_test.tolist())

# Model setup
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)

In [ ]:
# Hyperparameters
learning_rate = 2e-5
epochs = 3
batch_size = 8
accumulation_steps = 4
optimizer = AdamW(model.parameters(), lr=learning_rate)
scaler = GradScaler()

In [ ]:
# Training Loop
def train(model, data_loader):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(data_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with autocast():
            outputs = model(**batch)
            loss = outputs.loss / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps

    return total_loss / len(data_loader)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
for epoch in range(epochs):
    loss = train(model, train_loader)
    print(f'Epoch {epoch + 1}, Loss: {loss}')

In [ ]:
# Evaluation Function
def evaluate(model, data_loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast():
                outputs = model(**batch)
                logits = outputs.logits
                pred = torch.argmax(logits, dim=-1)
                predictions.extend(pred.cpu().numpy())
                true_labels.extend(batch['labels'].cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')
    return precision, recall, f1

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
precision, recall, f1 = evaluate(model, test_loader)
print(f'Precision: {precision}, Recall: {recall}, F1 Score: {f1}')

In [ ]:
def evaluate(model, data_loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            pred = torch.argmax(logits, dim=-1)
            predictions.extend(pred.cpu().numpy())
            true_labels.extend(batch['labels'].cpu().numpy())

            # Diagnostic outputs
            print("Batch Predictions:", pred)
            print("Batch Labels:", batch['labels'])

    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')
    return precision, recall, f1

# Assume test_loader is defined and set up properly
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)  # Define test_loader if not already defined
precision, recall, f1 = evaluate(model, test_loader)
print(f'Precision: {precision}, Recall: {recall}, F1 Score: {f1}')

In [ ]:
import torch.nn.functional as F

def evaluate_with_threshold(model, data_loader, threshold=0.5):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)[:, 1]  # Probabilities for the positive class
            pred = (probs > threshold).long()  # Apply threshold
            predictions.extend(pred.cpu().numpy())
            true_labels.extend(batch['labels'].cpu().numpy())

            print("Batch Probabilities:", probs)
            print("Batch Predictions:", pred)
            print("Batch Labels:", batch['labels'])

    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')
    return precision, recall, f1

# Adjust the threshold as needed
precision, recall, f1 = evaluate_with_threshold(model, test_loader, threshold=0.4)
print(f'Precision: {precision}, Recall: {recall}, F1 Score: {f1}')

In [ ]:
import csv

def evaluate_with_threshold_and_save_csv(model, data_loader, threshold=0.5, output_csv='model_output.csv'):
    model.eval()
    predictions, true_labels = [], []

    # Open the CSV file
    with torch.no_grad(), open(output_csv, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Batch Probabilities', 'Batch Predictions', 'Batch Labels'])

        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            probs = torch.nn.functional.softmax(logits, dim=-1)[:, 1]
            pred = (probs > threshold).long()
            predictions.extend(pred.cpu().numpy())
            true_labels.extend(batch['labels'].cpu().numpy())

            # Writing batch results to CSV within the context
            writer.writerow([probs.tolist(), pred.tolist(), batch['labels'].tolist()])

        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')

        # Write summary metrics to CSV within the context
        writer.writerow(['Metrics', 'Value'])
        writer.writerow(['Precision', precision])
        writer.writerow(['Recall', recall])
        writer.writerow(['F1 Score', f1])

    return precision, recall, f1

# Example usage
precision, recall, f1 = evaluate_with_threshold_and_save_csv(model, test_loader, threshold=0.4, output_csv='/content/model_output.csv')
